# Stage 1 (RT-DETR) inference on VinDr-CXR

Runs the RSNA-trained RT-DETR-L on VinDr-CXR images and writes `vindr_proposals.csv` with columns
`patientId, confidence, pred_x1, pred_y1, pred_x2, pred_y2` for the RAD-DINO external-evaluation notebook.

Attach as inputs: (1) your RT-DETR output containing the same checkpoint used for the RSNA proposals, (2) the VinDr-CXR images (png/jpg).
Edit only the config cell. Detector settings (`conf=0.15`, `imgsz=640`) match the RSNA harvest cell. Boxes are in the pixel space of each image file
(RT-DETR stretches to 640x640 internally and maps boxes back), so `box_coord_size` stays `None` downstream.

Make sure VinDr images got the same preprocessing as the RSNA images used in Step 1 (windowing/contrast, MONOCHROME1 inversion).

In [1]:
!pip install -q ultralytics opencv-python pandas numpy tqdm

In [2]:
!find /kaggle/input -path "*train_run/weights/*.pt"

/kaggle/input/notebooks/prahlad214/rt-detr-pipeline/experiments/baseline_rtdetr/train_run/weights/best.pt
/kaggle/input/notebooks/prahlad214/rt-detr-pipeline/experiments/baseline_rtdetr/train_run/weights/last.pt


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import RTDETR

WEIGHTS = "/kaggle/input/notebooks/prahlad214/rt-detr-pipeline/experiments/baseline_rtdetr/train_run/weights/best.pt"
IMAGE_DIR = "/kaggle/input/datasets/xhlulu/vinbigdata-chest-xray-resized-png-1024x1024/train"
ANNOTATIONS_CSV = None   # optional: competition train.csv, restricts to annotated images (not attached yet)
OUT_CSV = "/kaggle/working/vindr_proposals.csv"
CONF, IMGSZ, BATCH = 0.15, 640, 16      # CONF/IMGSZ identical to the RSNA harvest cell
MAX_IMAGES = None                        # smoke test; set to None for the full run

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [4]:
EXTS = (".png", ".jpg", ".jpeg")
paths = {}
for f in os.listdir(IMAGE_DIR):
    stem, ext = os.path.splitext(f)
    if ext.lower() in EXTS:
        paths[stem] = os.path.join(IMAGE_DIR, f)

if ANNOTATIONS_CSV:
    keep = set(pd.read_csv(ANNOTATIONS_CSV)["image_id"].astype(str))
    paths = {k: v for k, v in paths.items() if k in keep}
    print(f"restricted to annotated image ids: {len(paths)} (annotation ids: {len(keep)})")

ids = sorted(paths)[:MAX_IMAGES]
assert ids, "no images found"
print(f"images to process: {len(ids)}")

images to process: 15000


In [5]:
model = RTDETR(WEIGHTS)
device = 0 if torch.cuda.is_available() else "cpu"

rows, n_unreadable = [], 0
for s in tqdm(range(0, len(ids), BATCH), desc="stage1"):
    chunk, imgs = [], []
    for pid in ids[s:s + BATCH]:
        img = cv2.imread(paths[pid])          # BGR 3-channel, same reader as the RSNA harvest cell
        if img is None:
            n_unreadable += 1
            continue
        chunk.append(pid); imgs.append(img)
    if not imgs:
        continue
    # RT-DETR stretches each image to IMGSZ x IMGSZ and maps boxes back to the ORIGINAL file's pixel space
    results = model.predict(source=imgs, conf=CONF, imgsz=IMGSZ, device=device, verbose=False)
    for pid, img, r in zip(chunk, imgs, results):
        H, W = img.shape[:2]
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        for (x1, y1, x2, y2), c in zip(xyxy, conf):
            x1, x2 = np.clip([x1, x2], 0, W); y1, y2 = np.clip([y1, y2], 0, H)
            rows.append((pid, float(c), float(x1), float(y1), float(x2), float(y2)))

stage1:   0%|          | 0/938 [00:00<?, ?it/s]

In [6]:
out = pd.DataFrame(rows, columns=["patientId", "confidence", "pred_x1", "pred_y1", "pred_x2", "pred_y2"])
out.to_csv(OUT_CSV, index=False)

# ---- checks against what the RAD-DINO notebook requires ----
ok = out[(out.confidence >= CONF) & (out.pred_x2 > out.pred_x1) & (out.pred_y2 > out.pred_y1)]
n_img = out.patientId.nunique()
print(f"saved {OUT_CSV}: {len(out)} boxes ({len(ok)} usable) | images with >=1 proposal {n_img}/{len(ids)} "
      f"| proposals/image {len(ok) / max(n_img, 1):.2f} | unreadable {n_unreadable}")
assert out.patientId.isin(paths).all()
assert out.confidence.between(0, 1).all()
print("conf quantiles:", out.confidence.quantile([0, .25, .5, .75, 1]).round(3).tolist())

saved /kaggle/working/vindr_proposals.csv: 8361 boxes (8361 usable) | images with >=1 proposal 3467/15000 | proposals/image 2.41 | unreadable 0
conf quantiles: [0.15, 0.171, 0.204, 0.262, 0.652]


In [7]:
import pandas as pd
ref = pd.read_csv("/kaggle/input/notebooks/prahlad214/rt-detr-pipeline/test_candidate_proposals_epoch20.csv")
print("RSNA images with a proposal:", ref.patientId.nunique(), "/ 4003")
print("RSNA proposals per such image:", round(len(ref) / ref.patientId.nunique(), 2))
print("RSNA conf quantiles:", ref.confidence.quantile([0, .25, .5, .75, 1]).round(3).tolist())

RSNA images with a proposal: 2018 / 4003
RSNA proposals per such image: 5.0
RSNA conf quantiles: [0.15, 0.179, 0.225, 0.312, 0.65]
